In [1]:
import os
from pathlib import Path
import sys

# Automatically find repo root by looking for .git
ROOT = Path.cwd()
while not (ROOT / ".git").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

# FIX: fail loudly instead of silently falling back to cwd — otherwise every
# src.* import below fails with a confusing ModuleNotFoundError if this
# notebook is ever opened from outside the repo.
if not (ROOT / ".git").exists():
    raise RuntimeError(
        f"Could not find repo root from {Path.cwd()}. "
        "Open this notebook from inside the project directory."
    )

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

# Change the working directory to the repo root
os.chdir(ROOT)

In [2]:
import pandas as pd

In [3]:
from src.utils.data_loaders.read_settings_json import read_settings_json

args = read_settings_json()
args

{'Config': {'debug_mode': 'False', 'TEMP_CACHE': 'data/temp_cache'},
 'TrainingInput': {'CHART_OF_ACCOUNTS': 'data/training_input/chart_of_accounts.xlsx',
  'ENROLLEES': 'data/training_input/enrollees_pseudonymized.xlsx',
  'REVENUES': 'data/training_input/revenues_pseudonymized.xlsx'},
 'Training': {'MODEL_PARAMETERS': 'src/modules/machine_learning/parameters.json',
  'RESULTS_ROOT': 'data/training_results',
  'LOGS': 'data/training_logs',
  'DEPLOYED_MODELS': 'data/training_results/deployed_models',
  'observation_end': '2026/05/09',
  'target_feature': 'dtp_bracket',
  'test_size': '0.30'}}

In [4]:
# FIX: python-calamine isn't part of the standard pandas install set —
# fall back to openpyxl so the notebook doesn't hard-fail on a missing
# optional dependency.
try:
    df_revenues = pd.read_excel(args['TrainingInput']['REVENUES'], engine='calamine')
except ImportError:
    df_revenues = pd.read_excel(args['TrainingInput']['REVENUES'])

In [5]:
# FIX: same calamine fallback as the revenues cell above.
try:
    df_enrollees = pd.read_excel(args['TrainingInput']['ENROLLEES'], engine='calamine')
except ImportError:
    df_enrollees = pd.read_excel(args['TrainingInput']['ENROLLEES'])

In [6]:
from src.modules.feature_engineering.credit_sales_machine_learning import CreditSalesProcessor

# FIX: drop_missing_dtp=False left dtp_1..4 (and several derived columns:
# dtp_avg, early_payer_flag, dtp_rolling_std, dtp_max, ...) as NaN for
# invoices with no prior payment history. self.scaler.transform() lets NaN
# through silently, but the downstream CoxnetSurvivalAnalysis call inside
# generate_survival_features rejects NaN and raises "Input X contains NaN."
# Every training-time call site (step_3.py, step_5.py, the training
# notebook) uses drop_missing_dtp=True, so the deployed model was never
# fit on rows lacking DTP history -- scoring with True keeps the inference
# batch in-distribution with what the model actually learned.
cs_test = CreditSalesProcessor(df_revenues, df_enrollees, args,
                      drop_fully_paid_invoices=True,
                      drop_back_account_transactions=True,
                      calculate_payment_amounts=True,
                      add_description=True,
                      drop_missing_dtp=True)
df_cs_test = cs_test.show_data()
df_cs_test

Single due date records:   11151
Multiple due date records: 289
Dropped 9896 fully paid invoices. Remaining: 511
Dropped 442 invoices with missing DTP values. Remaining: 69


,school_year,student_id_pseudonimized,category_name,gross_receivables,amount_discounted,adjustments,credit_sale_amount,due_date,date_fully_paid,prepayments,...,due_quarter,opening_balance_flag,payment_ratio,early_payer_flag,on_time_streak,prev_bracket,dtp_rolling_std,dtp_max,plan_type_risk_score,description
10387,2026,0RA26SW3,Summer Worksheet,450.0,0.0,0.0,450.0,2026-03-24,NaT,0.0,...,1,1,0.864962,0.0,0,1.0,15.649814,28,2,Summer Worksheet
10410,2026,1VPCMYOP,G03-Books,5349.0,0.0,0.0,5349.0,2026-04-28,NaT,0.0,...,2,0,1.204526,1.0,4,0.0,24.198829,-1,1,Books (G03)
8237,2025,28D878FF,Disturbance Charges,1000.0,0.0,0.0,1000.0,2025-07-02,NaT,0.0,...,3,0,1.064608,0.0,0,3.0,59.494397,204,0,Penalties for late enrollee
8242,2025,2HLQMS3N,Disturbance Charges,1000.0,0.0,0.0,1000.0,2025-07-07,NaT,0.0,...,3,1,0.907981,0.0,0,3.0,44.040701,300,0,Penalties for late enrollee
8598,2025,2PPCAE8X,G08-Books,6680.0,0.0,0.0,6680.0,2025-08-08,NaT,0.0,...,3,1,0.942692,0.0,0,1.0,44.814432,7,0,Books (G08)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7272,2024,XGP1RBRH,SpE-OF-2nd,4267.0,0.0,0.0,4267.0,2024-12-10,NaT,0.0,...,4,1,0.930757,1.0,1,0.0,65.24569,17,0,Miscellaneous fees - 2 of 3 payments
9964,2025,XNTCIHFL,Events - Foundation Day,400.0,0.0,0.0,400.0,2026-03-05,NaT,0.0,...,1,0,1.240907,1.0,2,0.0,3.201562,3,1,Foundation Day
4761,2023,XRRBY1N0,Events - Tour,1960.0,0.0,0.0,1960.0,2023-12-06,NaT,0.0,...,4,1,0.877134,0.0,0,3.0,66.429787,188,0,Tour
5860,2023,XRRBY1N0,Events - Film Showing,400.0,0.0,0.0,400.0,2024-05-20,NaT,0.0,...,2,1,0.898068,0.0,0,NaN,45.90207,164,0,Film Showing


In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s %(message)s")

from src.modules.machine_learning.utils.inference.inference_pipeline import (
    find_deployed_model,
    load_inference_pipeline,
    run_batch_inference,
)

MODEL_DIR = args["Training"]["DEPLOYED_MODELS"]
SHOW_PROBA_COLS = True  # set True to include prob_* columns in display

In [8]:
# Locate and inspect the deployed artifact.
# Raises ValueError with upgrade instructions if artifact is pre-InferencePipeline.
artifact_path = find_deployed_model(MODEL_DIR)
print("Artifact:", artifact_path)

try:
    pipeline = load_inference_pipeline(MODEL_DIR)
    print(pipeline)
except ValueError as e:
    msg = str(e)
    print("[WARN] Could not load InferencePipeline:")
    print(" ", msg)
    print()
    print("Action required: Re-run Step 5 (Model Finalization) in the app to regenerate the artifact.")
    pipeline = None


INFO src.modules.machine_learning.utils.inference.inference_pipeline Loading InferencePipeline from data\training_results\deployed_models\finalized_two_stage_xgb_ada.pkl


Artifact: data\training_results\deployed_models\finalized_two_stage_xgb_ada.pkl


INFO src.modules.machine_learning.utils.inference.inference_pipeline Loaded InferencePipeline(
  model_key         = 'two_stage_xgb_ada'
  lda_transformer   = None
  time_points       = 9 points
  classes           = [np.str_('30_days'), np.str_('60_days'), np.str_('90_days'), np.str_('on_time')]
  feature_metadata  = ['plan_risk_map']
)


InferencePipeline(
  model_key         = 'two_stage_xgb_ada'
  lda_transformer   = None
  time_points       = 9 points
  classes           = [np.str_('30_days'), np.str_('60_days'), np.str_('90_days'), np.str_('on_time')]
  feature_metadata  = ['plan_risk_map']
)


In [ ]:
if pipeline is not None:
    # Select only numeric ML features from df_cs_test (drop non-numeric / label cols)
    EXCLUDE = {"dtp_bracket", "date_fully_paid", "due_date",
               "school_year", "student_id_pseudonimized", "category_name", "description"}
    X_infer = df_cs_test.drop(columns=[c for c in EXCLUDE if c in df_cs_test.columns])

    df_preds = run_batch_inference(
        input_source=X_infer,
        model_dir=MODEL_DIR,
        batch_size=1024,
        return_proba=True,
    )

    prob_cols = [c for c in df_preds.columns if c.startswith("prob_")]
    if SHOW_PROBA_COLS and not prob_cols:
        print("[WARN] SHOW_PROBA_COLS=True but no prob_* columns are present "
              "(return_proba=False was passed to run_batch_inference).")
    display_cols = [
        c for c in df_preds.columns
        if c not in prob_cols or SHOW_PROBA_COLS
    ]
    display(df_preds[display_cols].head(10))
else:
    df_preds = None
    print("Skipping inference -- no valid InferencePipeline loaded.")


In [ ]:
if df_preds is not None:
    df_preds.insert(
        df_preds.columns.get_loc("predicted_label") + 1,
        "actual_label",
        df_cs_test["dtp_bracket"].reindex(df_preds.index),
    )


In [10]:
if df_preds is not None:
    print("Predicted label distribution:")
    display(df_preds["predicted_label"].value_counts().rename("count").to_frame())
    n = len(df_preds)
    mk = df_preds["model_key"].iloc[0]
    ts = df_preds["run_timestamp"].iloc[0]
    print()
    print("Total rows scored:", n)
    print("Model key        :", mk)
    print("Run timestamp    :", ts)


Predicted label distribution:


,count
predicted_label,
on_time,32
60_days,16
90_days,14
30_days,7



Total rows scored: 69
Model key        : two_stage_xgb_ada
Run timestamp    : 2026-06-16T08:24:17.251060+00:00
